<a href="https://colab.research.google.com/github/hamdikhasawneh/AI-sepsis/blob/main/notebooks/05_User_Interface" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gradio pandas numpy scikit-learn joblib

In [ ]:
import gradio as gr
import pandas as pd

FEATURE_ORDER = [
    "heart_rate",
    "respiratory_rate",
    "temperature",
    "spo2",
    "systolic_bp",
    "diastolic_bp",
    "wbc",
    "lactate",
    "creatinine",
    "platelets",
    "age",
]

def clamp(value, low, high):
    return max(low, min(high, value))

def build_patient_df(
    heart_rate,
    respiratory_rate,
    temperature,
    spo2,
    systolic_bp,
    diastolic_bp,
    wbc,
    lactate,
    creatinine,
    platelets,
    age,
):
    return pd.DataFrame([{
        "heart_rate": heart_rate,
        "respiratory_rate": respiratory_rate,
        "temperature": temperature,
        "spo2": spo2,
        "systolic_bp": systolic_bp,
        "diastolic_bp": diastolic_bp,
        "wbc": wbc,
        "lactate": lactate,
        "creatinine": creatinine,
        "platelets": platelets,
        "age": age,
    }])[FEATURE_ORDER]

def demo_sepsis_logic(df):
    hr = df.loc[0, "heart_rate"]
    rr = df.loc[0, "respiratory_rate"]
    temp = df.loc[0, "temperature"]
    spo2 = df.loc[0, "spo2"]
    sbp = df.loc[0, "systolic_bp"]
    dbp = df.loc[0, "diastolic_bp"]
    wbc = df.loc[0, "wbc"]
    lactate = df.loc[0, "lactate"]
    creatinine = df.loc[0, "creatinine"]
    platelets = df.loc[0, "platelets"]
    age = df.loc[0, "age"]

    score = 0.08
    findings = []

    if temp >= 38.0 or temp <= 36.0:
        score += 0.12
        findings.append("Abnormal temperature")
    if hr >= 100:
        score += 0.10
        findings.append("Tachycardia")
    if rr >= 22:
        score += 0.12
        findings.append("Tachypnea")
    if spo2 < 92:
        score += 0.10
        findings.append("Low oxygen saturation")
    if sbp < 100:
        score += 0.14
        findings.append("Low systolic blood pressure")
    if dbp < 60:
        score += 0.05
        findings.append("Low diastolic blood pressure")
    if wbc > 12 or wbc < 4:
        score += 0.12
        findings.append("Abnormal white blood cell count")
    if lactate >= 2.0:
        score += 0.18
        findings.append("Elevated lactate")
    if creatinine >= 1.2:
        score += 0.08
        findings.append("Elevated creatinine")
    if platelets < 150:
        score += 0.08
        findings.append("Low platelet count")
    if age >= 65:
        score += 0.06
        findings.append("Older age risk factor")

    probability = clamp(score, 0.0, 0.99)

    if probability >= 0.75:
        risk_level = "High Sepsis Risk"
        color = "#dc2626"
        badge_bg = "#fee2e2"
        border = "#fca5a5"
        status = "🔴 Critical"
        recommendation = "Multiple risk indicators are elevated. Immediate clinical review is recommended."
    elif probability >= 0.45:
        risk_level = "Moderate Sepsis Risk"
        color = "#d97706"
        badge_bg = "#ffedd5"
        border = "#fdba74"
        status = "🟡 Needs Attention"
        recommendation = "Several warning signs are present. Further monitoring and prompt assessment are advised."
    else:
        risk_level = "Low Sepsis Risk"
        color = "#16a34a"
        badge_bg = "#dcfce7"
        border = "#86efac"
        status = "🟢 Stable"
        recommendation = "Current indicators suggest lower risk based on the entered values."

    if not findings:
        findings = ["No major demo-rule abnormalities detected"]

    top_findings = "".join([
        f'<span class="finding-chip">{item}</span>'
        for item in findings[:6]
    ])

    card_html = f"""
    <div class="result-card" style="border-left: 8px solid {color};">
        <div class="result-header">
            <div>
                <div class="result-label">Sepsis Screening Result</div>
                <div class="result-title" style="color:{color};">{risk_level}</div>
            </div>
            <div class="status-badge" style="background:{badge_bg}; border:1px solid {border}; color:{color};">
                {status}
            </div>
        </div>

        <div class="result-metrics">
            <div class="metric-box">
                <div class="metric-label">Probability</div>
                <div class="metric-value">{round(probability * 100, 2)}%</div>
            </div>
            <div class="metric-box">
                <div class="metric-label">Recommendation</div>
                <div class="metric-text">{recommendation}</div>
            </div>
        </div>

        <div class="findings-box">
            <div class="findings-title">Key Indicators</div>
            <div>{top_findings}</div>
        </div>
    </div>
    """

    summary_md = f"""
### Clinical Interpretation

**Risk Level:** {risk_level}
**Probability:** {round(probability * 100, 2)}%
**Status:** {status}

**Recommendation:**
{recommendation}

**Important:** This is currently a demo interface using temporary logic and should not be used for real medical diagnosis.
"""

    return risk_level, round(probability * 100, 2), status, recommendation, card_html, summary_md

def predict_sepsis(
    heart_rate,
    respiratory_rate,
    temperature,
    spo2,
    systolic_bp,
    diastolic_bp,
    wbc,
    lactate,
    creatinine,
    platelets,
    age,
):
    values = [
        heart_rate, respiratory_rate, temperature, spo2,
        systolic_bp, diastolic_bp, wbc, lactate,
        creatinine, platelets, age
    ]

    if any(v is None for v in values):
        empty_df = pd.DataFrame(columns=FEATURE_ORDER)
        return (
            "Missing Input",
            0.0,
            "⚠️ Incomplete",
            "Please complete all fields before prediction.",
            """
            <div class="result-card" style="border-left: 8px solid #f97316;">
                <div class="result-title" style="color:#c2410c;">Missing Input</div>
                <div class="metric-text" style="margin-top:10px;">
                    Please fill in all patient fields before running the prediction.
                </div>
            </div>
            """,
            "### Clinical Interpretation\nPlease complete all fields before prediction.",
            empty_df
        )

    df = build_patient_df(
        heart_rate, respiratory_rate, temperature, spo2,
        systolic_bp, diastolic_bp, wbc, lactate,
        creatinine, platelets, age
    )

    risk_level, probability, status, recommendation, card_html, summary_md = demo_sepsis_logic(df)

    return (
        risk_level,
        probability,
        status,
        recommendation,
        card_html,
        summary_md,
        df
    )

def example_high():
    return [118, 26, 39.1, 89, 92, 56, 16.4, 3.5, 1.7, 128, 74]

def example_moderate():
    return [101, 22, 38.1, 94, 101, 64, 12.8, 2.1, 1.1, 165, 61]

def example_low():
    return [82, 17, 36.9, 98, 120, 78, 7.4, 1.0, 0.8, 240, 34]

def clear_all():
    empty_df = pd.DataFrame(columns=FEATURE_ORDER)
    return [None] * 11 + ["", 0.0, "", "", "", "", empty_df]

custom_css = """
body, .gradio-container {
    background: linear-gradient(180deg, #f4fbff 0%, #eef9ff 45%, #f3fcf8 100%);
    font-family: Arial, sans-serif;
}

.main-wrap {
    max-width: 1200px;
    margin: 0 auto;
}

.app-shell {
    background: rgba(255,255,255,0.88);
    border: 1px solid #d9f0ff;
    border-radius: 24px;
    padding: 24px;
    box-shadow: 0 10px 30px rgba(125, 180, 220, 0.12);
    backdrop-filter: blur(8px);
}

.hero-card {
    background: linear-gradient(135deg, #7dd3fc 0%, #93c5fd 50%, #bae6fd 100%);
    color: white;
    border-radius: 24px;
    padding: 28px;
    box-shadow: 0 12px 32px rgba(147, 197, 253, 0.25);
    margin-bottom: 18px;
}

.hero-title {
    font-size: 34px;
    font-weight: 800;
    margin-bottom: 8px;
}

.hero-subtitle {
    font-size: 16px;
    opacity: 0.95;
    line-height: 1.6;
}

.info-banner {
    background: #fff7ed;
    border: 1px solid #fdba74;
    color: #9a3412;
    border-radius: 18px;
    padding: 14px 16px;
    margin-bottom: 18px;
    font-size: 15px;
}

.stat-grid {
    display: grid;
    grid-template-columns: repeat(3, 1fr);
    gap: 14px;
    margin: 18px 0 10px 0;
}

.stat-card {
    background: rgba(255,255,255,0.18);
    border: 1px solid rgba(255,255,255,0.28);
    border-radius: 18px;
    padding: 16px;
}

.stat-label {
    font-size: 13px;
    opacity: 0.9;
    margin-bottom: 6px;
}

.stat-value {
    font-size: 22px;
    font-weight: 700;
}

.section-title {
    font-size: 22px;
    font-weight: 700;
    color: #0f172a;
    margin: 16px 0 8px 0;
}

.result-card {
    background: white;
    border-radius: 22px;
    padding: 20px;
    box-shadow: 0 6px 18px rgba(15, 23, 42, 0.08);
    margin-top: 8px;
}

.result-header {
    display: flex;
    justify-content: space-between;
    align-items: flex-start;
    gap: 12px;
    margin-bottom: 16px;
}

.result-label {
    color: #64748b;
    font-size: 14px;
    margin-bottom: 6px;
}

.result-title {
    font-size: 30px;
    font-weight: 800;
    line-height: 1.2;
}

.status-badge {
    padding: 10px 14px;
    border-radius: 999px;
    font-weight: 700;
    white-space: nowrap;
}

.result-metrics {
    display: grid;
    grid-template-columns: 220px 1fr;
    gap: 14px;
    margin-bottom: 14px;
}

.metric-box {
    background: #f8fbff;
    border: 1px solid #dbeafe;
    border-radius: 16px;
    padding: 14px;
}

.metric-label {
    color: #64748b;
    font-size: 13px;
    margin-bottom: 6px;
}

.metric-value {
    font-size: 30px;
    font-weight: 800;
    color: #0f172a;
}

.metric-text {
    font-size: 15px;
    color: #1e293b;
    line-height: 1.6;
}

.findings-box {
    background: #f8fbff;
    border: 1px dashed #bfdbfe;
    border-radius: 16px;
    padding: 14px;
}

.findings-title {
    font-size: 15px;
    font-weight: 700;
    color: #0f172a;
    margin-bottom: 10px;
}

.finding-chip {
    display: inline-block;
    background: #e0f2fe;
    color: #075985;
    border: 1px solid #bae6fd;
    border-radius: 999px;
    padding: 7px 12px;
    margin: 0 8px 8px 0;
    font-size: 13px;
    font-weight: 600;
}

footer {display:none !important;}

@media (max-width: 900px) {
    .stat-grid {
        grid-template-columns: 1fr;
    }
    .result-metrics {
        grid-template-columns: 1fr;
    }
}
"""

with gr.Blocks(css=custom_css, title="AI Sepsis Detection Dashboard") as demo:
    gr.HTML("""
    <div class="main-wrap">
        <div class="hero-card">
            <div class="hero-title">AI Sepsis Detection Dashboard</div>
            <div class="hero-subtitle">
                Smart clinical screening interface for nurses and doctors using patient vital signs
                and laboratory indicators.
            </div>
            <div class="stat-grid">
                <div class="stat-card">
                    <div class="stat-label">System Mode</div>
                    <div class="stat-value">Demo</div>
                </div>
                <div class="stat-card">
                    <div class="stat-label">Access</div>
                    <div class="stat-value">Doctors / Nurses</div>
                </div>
                <div class="stat-card">
                    <div class="stat-label">Prediction Type</div>
                    <div class="stat-value">Sepsis Risk</div>
                </div>
            </div>
        </div>

        <div class="info-banner">
            <b>Demo Notice:</b> This version uses temporary rule-based logic until your trained ML model is added.
        </div>

        <div class="app-shell">
    """)

    with gr.Row():
        with gr.Column(scale=1):
            gr.HTML('<div class="section-title">Patient Vitals</div>')
            with gr.Group():
                heart_rate = gr.Number(label="Heart Rate (bpm)", value=110)
                respiratory_rate = gr.Number(label="Respiratory Rate", value=24)
                temperature = gr.Number(label="Temperature (°C)", value=38.6)
                spo2 = gr.Number(label="SpO₂ (%)", value=92)
                systolic_bp = gr.Number(label="Systolic BP (mmHg)", value=96)
                diastolic_bp = gr.Number(label="Diastolic BP (mmHg)", value=60)

        with gr.Column(scale=1):
            gr.HTML('<div class="section-title">Laboratory Results</div>')
            with gr.Group():
                wbc = gr.Number(label="WBC (x10⁹/L)", value=14.8)
                lactate = gr.Number(label="Lactate (mmol/L)", value=2.7)
                creatinine = gr.Number(label="Creatinine (mg/dL)", value=1.4)
                platelets = gr.Number(label="Platelets (x10⁹/L)", value=140)
                age = gr.Number(label="Age", value=68)

    gr.HTML('<div class="section-title">Quick Actions</div>')
    with gr.Row():
        predict_btn = gr.Button("Run Prediction", variant="primary")
        high_btn = gr.Button("Load High-Risk Example")
        moderate_btn = gr.Button("Load Moderate-Risk Example")
        low_btn = gr.Button("Load Low-Risk Example")
        clear_btn = gr.Button("Clear")

    gr.HTML('<div class="section-title">Prediction Output</div>')
    with gr.Row():
        risk_level_out = gr.Textbox(label="Risk Level")
        probability_out = gr.Number(label="Probability (%)")
        status_out = gr.Textbox(label="Status")

    recommendation_out = gr.Textbox(label="Recommendation")
    result_card_out = gr.HTML(label="Result Card")
    interpretation_out = gr.Markdown()
    patient_df_out = gr.Dataframe(
        label="Submitted Patient Data",
        headers=FEATURE_ORDER,
        datatype="number",
        interactive=False
    )

    predict_btn.click(
        fn=predict_sepsis,
        inputs=[
            heart_rate, respiratory_rate, temperature, spo2,
            systolic_bp, diastolic_bp, wbc, lactate,
            creatinine, platelets, age
        ],
        outputs=[
            risk_level_out,
            probability_out,
            status_out,
            recommendation_out,
            result_card_out,
            interpretation_out,
            patient_df_out
        ]
    )

    high_btn.click(
        fn=example_high,
        outputs=[
            heart_rate, respiratory_rate, temperature, spo2,
            systolic_bp, diastolic_bp, wbc, lactate,
            creatinine, platelets, age
        ]
    )

    moderate_btn.click(
        fn=example_moderate,
        outputs=[
            heart_rate, respiratory_rate, temperature, spo2,
            systolic_bp, diastolic_bp, wbc, lactate,
            creatinine, platelets, age
        ]
    )

    low_btn.click(
        fn=example_low,
        outputs=[
            heart_rate, respiratory_rate, temperature, spo2,
            systolic_bp, diastolic_bp, wbc, lactate,
            creatinine, platelets, age
        ]
    )

    clear_btn.click(
        fn=clear_all,
        outputs=[
            heart_rate, respiratory_rate, temperature, spo2,
            systolic_bp, diastolic_bp, wbc, lactate,
            creatinine, platelets, age,
            risk_level_out, probability_out, status_out,
            recommendation_out, result_card_out,
            interpretation_out, patient_df_out
        ]
    )

    gr.HTML("</div></div>")

demo.launch(
    share=True,
    auth=[
        ("doctor", "doctor123"),
        ("nurse", "nurse123"),
    ]
)

/tmp/ipykernel_159/2985857463.py:432: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css, title="AI Sepsis Detection Dashboard") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d58b6c52aed2545b20.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
